# Notebook 05: Hedging Backtest

**Purpose**: demonstrate the canonical historical hedge study used by this repository.

This notebook focuses on the evaluation design, not just on producing one PnL number. The canonical hedge study now:

1. selects fixed listed contracts at trade entry,
2. uses the observed market premium for both models,
3. recalibrates the arbitrage-free surface on each trade date,
4. derives a fresh local-vol surface on each trade date,
5. and reports entry pricing bias separately from replication error and net market-trade PnL.


### Why This Notebook Changed

An older style of backtest calibrates one surface once and then replays hedging on underlying history. That can be informative, but it mixes together too many effects:

- a model may simply overprice or underprice the option at entry,
- the chosen strike may drift if it is defined as `K = S_0` path by path,
- and a surface calibrated on one date may be unfairly tested on very different market regimes.

The repository therefore treats the **daily market-panel hedge study** as the canonical evaluation. That is the workflow exercised in this notebook.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'notebook5_market_panel'
SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'run_backtest_market_panel.py'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SCRIPT_PATH exists:', SCRIPT_PATH.exists())


### Input Data For The Hedge Study

A real historical hedge study needs a **dated option panel**, meaning option-chain snapshots across many dates.

The repository now has a canonical Theta-backed daily panel workflow:

- Theta provides the full SPY option chain,
- the repo backfills recent market days at **15:45 ET**,
- the scheduled task appends one new dated snapshot each trading day,
- and the backtest consumes that dated panel directly.

So the notebook does the following:

- if a real processed panel exists, it uses that panel,
- otherwise it falls back to a synthetic Black-Scholes panel so the notebook still demonstrates the workflow.


In [ ]:
real_panel_candidates = [
    PROJECT_ROOT / 'data' / 'raw' / 'spy_options_panel.parquet',
    PROJECT_ROOT / 'data' / 'raw' / 'spy_options_panel.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'spy_options_panel.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'spy_options_panel.csv',
]
real_panel = next((p for p in real_panel_candidates if p.exists()), None)

if real_panel is not None:
    PANEL_FILE = real_panel
    HISTORY_FILE = PROJECT_ROOT / 'data' / 'raw' / 'spy_history.csv'
    TICKER = 'SPY'
    DATA_MODE = 'real_panel'
else:
    sys.path.append(str(PROJECT_ROOT))
    from tests.fixtures.synthetic_data import BSDailyPanelConfig, generate_bs_daily_panel

    panel_df, history_df = generate_bs_daily_panel(BSDailyPanelConfig())
    PANEL_FILE = OUTPUT_DIR / 'synthetic_option_panel.csv'
    HISTORY_FILE = OUTPUT_DIR / 'synthetic_history.csv'
    panel_df.to_csv(PANEL_FILE, index=False)
    history_df.to_csv(HISTORY_FILE, index=False)
    TICKER = 'SYNTH'
    DATA_MODE = 'synthetic_demo'

print('DATA_MODE:', DATA_MODE)
print('PANEL_FILE:', PANEL_FILE)
print('HISTORY_FILE:', HISTORY_FILE)


### Contract Selection And Backtest Parameters

The script chooses fixed listed contracts from the dated panel. In the canonical setup, this means a liquid near-ATM short-dated SPY call family with explicit liquidity filters.

The important conceptual point is that the strike is chosen **once at trade entry** from the listed chain. It is not reset every day. That makes the hedge study behave like an actual option position rather than like a moving target.


In [ ]:
OPTION_TYPE = 'call'
STRIKE_MODE = 'atm'
TARGET_T = 8.0 / 365.0
ENTRY_STEP_DAYS = 20 if DATA_MODE == 'synthetic_demo' else 1
TX_BPS = 10.0 if DATA_MODE != 'synthetic_demo' else 0.0
MAX_CONTRACTS = 1 if DATA_MODE == 'synthetic_demo' else 10

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--ticker', TICKER,
    '--panel_file', str(PANEL_FILE),
    '--history_csv', str(HISTORY_FILE),
    '--out', str(OUTPUT_DIR),
    '--option_type', OPTION_TYPE,
    '--strike_mode', STRIKE_MODE,
    '--T', str(TARGET_T),
    '--entry_step_days', str(ENTRY_STEP_DAYS),
    '--max_contracts', str(MAX_CONTRACTS),
    '--tx_bps', str(TX_BPS),
    '--contract_moneyness_band', '0.02',
    '--contract_max_relative_spread', '0.05',
    '--contract_min_volume', '25',
    '--contract_min_open_interest', '200',
    '--min_volume', '25',
    '--min_open_interest', '200',
    '--moneyness_min', '0.90',
    '--moneyness_max', '1.10',
    '--max_relative_spread', '0.05',
    '--min_points_per_slice', '10',
    '--n_strikes', '45',
    '--pde_space', '120',
    '--pde_time', '100',
]

print('Running command:')
print(' '.join(cmd))


### What The Script Does Internally

For each entry date in the study, the script:

1. selects a fixed listed contract,
2. calibrates that day's arbitrage-free surface from the full option snapshot,
3. derives the local-vol surface for that date,
4. computes BS and Local Vol deltas for the same contract,
5. redoes that calibration on later dates,
6. and writes separate outputs for:
   - entry pricing error,
   - replication error under model premium,
   - and realized net market-trade PnL under the common observed market premium.

That separation is what makes the redesigned hedge study much more interpretable than the old single-number replay.


In [ ]:
proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in proc.stdout:
    print(line, end='')

if proc.wait() != 0:
    raise RuntimeError('Market-panel backtest failed')


### Reading The Outputs

The two model CSVs now contain richer information than the old backtest format. In particular, they include:

- `market_premium`: the observed entry price of the listed option,
- `model_entry_price`: the model's own entry valuation,
- `entry_pricing_error`: model price minus market premium,
- `replication_error_net`: the terminal replication error if the model had been sold at its own fair price,
- `market_pnl_net`: the realized net PnL when both models hedge the same market liability,
- and transaction-cost diagnostics.

Those columns let us answer a better question: is Local Vol losing because its deltas are poor, because it misprices entry, or both?


In [ ]:
sys.path.append(str(PROJECT_ROOT / 'src'))
from hedging.metrics import summarize_market_panel_backtest

bs = pd.read_csv(OUTPUT_DIR / 'daily_market_backtest_BS.csv')
lv = pd.read_csv(OUTPUT_DIR / 'daily_market_backtest_LocalVol.csv')
summary = json.loads((OUTPUT_DIR / 'daily_market_backtest_summary.json').read_text())

print('=== Daily market-panel summary JSON ===')
print(json.dumps(summary, indent=2))
print('\n=== Black-Scholes summary ===')
print(summarize_market_panel_backtest(bs))
print('\n=== LocalVol summary ===')
print(summarize_market_panel_backtest(lv))

cols = [
    'trade_id',
    'entry_date',
    'maturity_date',
    'strike',
    'market_premium',
    'model_entry_price',
    'entry_pricing_error',
    'replication_error_net',
    'market_pnl_net',
    'total_tx_cost',
]
print('\n=== BS trades ===')
print(bs[cols].head())
print('\n=== LocalVol trades ===')
print(lv[cols].head())


### Final Interpretation

This notebook is the last stage of the story because hedging is where every earlier stage becomes visible at once.

If the data cleaning is weak, the surface fit is unstable.
If the surface is not arbitrage-free, Dupire becomes unreliable.
If local volatility is numerically noisy, pricing and deltas degrade.
If the hedge study is poorly designed, pricing bias can be mistaken for hedge failure.

The redesigned market-panel backtest is meant to avoid that last mistake. It does not guarantee that Local Vol will beat Black-Scholes on real data. It does guarantee that the comparison is much more defensible.
